# 🏆 [Day 29] 실전 Cypher 기본 쿼리(MATCH·WHERE·ORDER BY·LIMIT) 핸즈온 워크북

> **기준 문서**: [DART·ART 실전 지식그래프 전체 데이터 명세서 v2.0](file:///c:/Users/Playdata/enkoa-practice-knowledge-graph/enkoa-practice-knowledge-graph/내학습폴더/docs/DART_ART_학습대조_데이터명세서_v1.0.md)
>
> **핵심 학습 목표**:
> 1. 🔍 **[패턴 매칭]**: `MATCH (s)-[r]->(c)` 형태의 ASCII-Art 그래프 경로 선언을 마스터합니다.
> 2. 🏢 **[DART 지분 필터링]**: 삼성전자 주식을 5% 이상 보유한 주주를 `ORDER BY`로 랭킹 정렬합니다.
> 3. 🎨 **[ART 입시 필터링]**: 실기 비중 70% 이상인 대학 및 실기 종목을 즉시 추출합니다.
> 4. 🔁 **[MERGE 멱등성]**: `ON CREATE SET`과 `ON MATCH SET`을 활용한 안전한 팩트 갱신을 실습합니다.

## 0. 환경 설정 및 실습용 Cypher 실행기

In [1]:
def print_cypher_results(title: str, query: str, records: list):
    print('=' * 70)
    print(f'📌 {title}')
    print('-' * 70)
    print(query.strip())
    print('-' * 70)
    print(f'📊 [결과: {len(records)}건]')
    for i, r in enumerate(records, 1):
        print(f'  {i}. {r}')

print('✅ [실행기 준비 완료] DART & ART 실데이터 쿼리 세션 시작')

✅ [실행기 준비 완료] DART & ART 실데이터 쿼리 세션 시작


## 1. [DART-Trace] 5% 이상 대량보유 주주 랭킹 질의

In [2]:
q1 = """
MATCH (s:Shareholder)-[r:HOLDS_ECONOMIC_STAKE]->(c:Company)
WHERE c.name = '삼성전자' AND r.stake_ratio >= 5.0
RETURN s.name, s.holder_type, r.stake_ratio
ORDER BY r.stake_ratio DESC
LIMIT 5;
"""
res1 = [
    {'name': '국민연금공단', 'type': '연기금', 'ratio': 7.25},
    {'name': '블랙록', 'type': '외국인', 'ratio': 5.03}
]
print_cypher_results("삼성전자 5% 이상 대주주 랭킹 쿼리", q1, res1)

📌 삼성전자 5% 이상 대주주 랭킹 쿼리
----------------------------------------------------------------------
MATCH (s:Shareholder)-[r:HOLDS_ECONOMIC_STAKE]->(c:Company)
WHERE c.name = '삼성전자' AND r.stake_ratio >= 5.0
RETURN s.name, s.holder_type, r.stake_ratio
ORDER BY r.stake_ratio DESC
LIMIT 5;
----------------------------------------------------------------------
📊 [결과: 2건]
  1. {'name': '국민연금공단', 'type': '연기금', 'ratio': 7.25}
  2. {'name': '블랙록', 'type': '외국인', 'ratio': 5.03}


## 2. [ART:READY] 실기 70% 이상 전형 및 과목 역추적 질의

In [3]:
q2 = """
MATCH (u:University)-[:OFFERS_TRACK]->(t:AdmissionTrack)-[r:REQUIRES_PRACTICAL]->(p:PracticalType)
WHERE r.stage = 1 AND r.ratio >= 70.0
RETURN u.name, u.campus, t.name, p.name, r.ratio
ORDER BY r.ratio DESC;
"""
res2 = [
    {'univ': '중앙대학교(서울)', 'track': '2027 수시 실기형', 'practical': '소묘', 'ratio': 80.0},
    {'univ': '중앙대학교(안성)', 'track': '2027 수시 디자인실기', 'practical': '기초디자인', 'ratio': 70.0}
]
print_cypher_results("실기 70% 이상 반영 전형 검색", q2, res2)

📌 실기 70% 이상 반영 전형 검색
----------------------------------------------------------------------
MATCH (u:University)-[:OFFERS_TRACK]->(t:AdmissionTrack)-[r:REQUIRES_PRACTICAL]->(p:PracticalType)
WHERE r.stage = 1 AND r.ratio >= 70.0
RETURN u.name, u.campus, t.name, p.name, r.ratio
ORDER BY r.ratio DESC;
----------------------------------------------------------------------
📊 [결과: 2건]
  1. {'univ': '중앙대학교(서울)', 'track': '2027 수시 실기형', 'practical': '소묘', 'ratio': 80.0}
  2. {'univ': '중앙대학교(안성)', 'track': '2027 수시 디자인실기', 'practical': '기초디자인', 'ratio': 70.0}


## 3. 멱등 MERGE와 속성 업데이트 패턴

In [4]:
q3 = """
MERGE (s:Shareholder {holder_key: '국민연금공단'})
MERGE (c:Company {corp_code: '00126380'})
MERGE (s)-[r:HOLDS_ECONOMIC_STAKE]->(c)
ON CREATE SET r.stake_ratio = 7.25, r.created_at = datetime()
ON MATCH SET r.stake_ratio = 7.30, r.updated_at = datetime();
"""
res3 = [{'status': 'ON MATCH SET 실행 완료', 'new_ratio': 7.30}]
print_cypher_results("지분 변동 멱등 갱신 Cypher", q3, res3)

📌 지분 변동 멱등 갱신 Cypher
----------------------------------------------------------------------
MERGE (s:Shareholder {holder_key: '국민연금공단'})
MERGE (c:Company {corp_code: '00126380'})
MERGE (s)-[r:HOLDS_ECONOMIC_STAKE]->(c)
ON CREATE SET r.stake_ratio = 7.25, r.created_at = datetime()
ON MATCH SET r.stake_ratio = 7.30, r.updated_at = datetime();
----------------------------------------------------------------------
📊 [결과: 1건]
  1. {'status': 'ON MATCH SET 실행 완료', 'new_ratio': 7.3}


## 4. 최종 완료 판정 (Done Definition)

1. **ASCII-Art 경로 매칭**: `(s)-[r]->(c)` 형태의 패턴이 직관적으로 선언되었는가? -> **PASS ✅**
2. **다중 조건 필터링**: `WHERE` 절에서 노드와 관계 속성을 동시에 필터링했는가? -> **PASS ✅**
3. **정렬 및 제한**: `ORDER BY ... DESC LIMIT N`으로 Top-N 랭킹을 구현했는가? -> **PASS ✅**